# LeetCode 663 — Equal Tree Partition bằng Python

Link đề: https://leetcode.com/problems/equal-tree-partition/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        ...
```

Ghi chú: đây là bài **Premium** (🔒). Đề trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại với đề gốc** nếu bạn có tài khoản Premium. Trong notebook này ta tự dựng `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode (`build_tree`) và hàm in cây ngược lại để chạy mọi ví dụ ngay tại đây.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xóa **đúng một cạnh** của cây nhị phân thì cây tách thành hai cây; hỏi có cách xóa sao cho **tổng giá trị** của hai cây bằng nhau không.
2. Nắm các công cụ Python cần dùng: lớp `TreeNode`, đệ quy **hậu thứ tự** (postorder) để tính tổng cây con, `collections.Counter`, ngăn xếp tường minh thay đệ quy, phép `%` với số âm và giới hạn đệ quy.
3. Đi từ **cắt thử từng cạnh và tính lại hai tổng (O(n²))** → **một lượt hậu thứ tự lưu mọi tổng cây con** → **bảng đếm `Counter` có xử lý riêng tổng bằng 0** → **bản lặp không đệ quy**.
4. Hiểu mấu chốt: cắt cạnh nối nút `v` với cha tách ra hai phần có tổng `sub(v)` và `S - sub(v)` (với `S` là tổng cả cây); chúng bằng nhau khi và chỉ khi `sub(v) = S / 2`. Vậy chỉ cần: **tồn tại nút `v` không phải gốc có tổng cây con bằng `S / 2`**.
5. Tránh các lỗi kinh điển: tính cả gốc (không có cạnh nào để cắt), quên kiểm tra `S` lẻ, chỉ xét hai con của gốc, và đệ quy quá sâu trên cây lệch 10^4 nút.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân. Trả về `True` nếu có thể **chia cây thành hai cây có tổng giá trị bằng nhau** bằng cách **xóa đúng một cạnh** của cây ban đầu; ngược lại trả `False`.

**Ví dụ 1**

- Input: `root = [5,10,10,null,null,2,3]`
- Output: `true`
- Giải thích: cắt cạnh giữa gốc `5` và nút `10` bên phải: cây còn lại có `5 + 10 = 15`, cây tách ra có `10 + 2 + 3 = 15`.

**Ví dụ 2**

- Input: `root = [1,2,10,null,null,2,20]`
- Output: `false`
- Giải thích: không có cạnh nào chia thành hai phần có tổng bằng nhau.

**Ràng buộc:**

- Số nút của cây thuộc đoạn `[1, 10^4]`.
- `-10^5 <= Node.val <= 10^5`

Lưu ý: giá trị có thể **âm** hoặc `0`; cây chỉ có một nút thì **không có cạnh nào** để xóa nên đáp án là `False`.

In [1]:
from typing import Optional, List
from collections import deque, Counter
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    # dựng cây từ danh sách theo thứ tự tầng, đúng quy ước của LeetCode (null không có con)
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([5, 10, 10, None, None, 2, 3], True),
    ([1, 2, 10, None, None, 2, 20], False),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", expected)

[5, 10, 10, None, None, 2, 3] -> kỳ vọng True
[1, 2, 10, None, None, 2, 20] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. `TreeNode` và dựng cây từ danh sách LeetCode

Mỗi nút có `val`, `left`, `right`. Danh sách đầu vào theo thứ tự tầng; `null` nghĩa là không có nút ở chỗ đó và **nút `null` không có con**. Hàm `build_tree` dùng hàng đợi `deque`: lấy từng nút ra, gán hai phần tử kế tiếp làm con trái/phải. In thử cây ví dụ 1 để nhìn hình dạng.

In [2]:
def show(node, indent=0, tag="gốc"):
    if node is None:
        return
    print("  " * indent + f"{tag}: {node.val}")
    show(node.left, indent + 1, "L")
    show(node.right, indent + 1, "R")

show(build_tree([5, 10, 10, None, None, 2, 3]))

gốc: 5
  L: 10
  R: 10
    L: 2
    R: 3


### 3.2. Tổng cây con và duyệt **hậu thứ tự** (postorder)

**Cây con** gốc `v` gồm `v` và mọi hậu duệ của nó. Tổng cây con thỏa hệ thức đệ quy:

`sub(v) = v.val + sub(v.left) + sub(v.right)`, với `sub(None) = 0`.

Muốn tính `sub(v)` ta cần `sub` của hai con **trước**, tức là xử lý con trước rồi mới tới cha: đó là duyệt **hậu thứ tự** (left, right, node). Nhờ vậy một lượt duyệt `O(n)` tính được tổng cây con của **mọi** nút.

In [3]:
def subtree_sums_postorder(root):
    # trả về danh sách (val, tổng cây con) theo thứ tự hậu thứ tự
    out = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        out.append((node.val, s))
        return s
    dfs(root)
    return out

print(subtree_sums_postorder(build_tree([5, 10, 10, None, None, 2, 3])))
# [(10, 10), (2, 2), (3, 3), (10, 15), (5, 30)]  -> nút cuối cùng là gốc, có tổng cả cây

[(10, 10), (2, 2), (3, 3), (10, 15), (5, 30)]


### 3.3. Xóa một cạnh = tách một cây con

Cây có `n` nút thì có đúng `n - 1` cạnh, và mỗi cạnh nối một nút **không phải gốc** `v` với cha của nó. Xóa cạnh đó tách ra hai cây:

- cây **con** gốc `v` với tổng `sub(v)`,
- phần **còn lại** (chứa gốc) với tổng `S - sub(v)`, trong đó `S = sub(root)` là tổng toàn cây.

Như vậy mỗi cạnh ứng với đúng một nút không phải gốc. Gốc **không** có cạnh nào "đi lên", nên không có cách cắt nào ứng với gốc.

In [4]:
root = build_tree([5, 10, 10, None, None, 2, 3])
sums = subtree_sums_postorder(root)
S = sums[-1][1]
print("tổng cả cây S =", S)
for val, sub in sums[:-1]:               # bỏ nút cuối (gốc)
    print(f"cắt cạnh tới nút {val}: phần tách ra = {sub}, phần còn lại = {S - sub}, bằng nhau? {sub == S - sub}")

tổng cả cây S = 30
cắt cạnh tới nút 10: phần tách ra = 10, phần còn lại = 20, bằng nhau? False
cắt cạnh tới nút 2: phần tách ra = 2, phần còn lại = 28, bằng nhau? False
cắt cạnh tới nút 3: phần tách ra = 3, phần còn lại = 27, bằng nhau? False
cắt cạnh tới nút 10: phần tách ra = 15, phần còn lại = 15, bằng nhau? True


### 3.4. Phép `%` với số âm và kiểm tra lẻ

Tổng `S` có thể âm. Trong Python `%` luôn cho kết quả **cùng dấu với số chia**: `-3 % 2 == 1` (khác C++/Java cho `-1`). Vì vậy kiểm tra "lẻ" bằng `S % 2 != 0` (hoặc `S & 1`) an toàn. Còn phép chia nguyên `//` làm tròn **xuống**: `-3 // 2 == -2`. Ta tránh rắc rối bằng cách so sánh `2 * sub == S`, không cần chia.

In [5]:
for s in [6, 7, -6, -7, 0]:
    print(f"S={s:3}: S % 2 = {s % 2}, lẻ? {s % 2 != 0}, S // 2 = {s // 2}")

S=  6: S % 2 = 0, lẻ? False, S // 2 = 3
S=  7: S % 2 = 1, lẻ? True, S // 2 = 3
S= -6: S % 2 = 0, lẻ? False, S // 2 = -3
S= -7: S % 2 = 1, lẻ? True, S // 2 = -4
S=  0: S % 2 = 0, lẻ? False, S // 2 = 0


### 3.5. Giới hạn đệ quy

Python mặc định cho phép đệ quy sâu khoảng `1000` tầng. Cây `10^4` nút có thể là một **dây chuyền** (mỗi nút một con) sâu `10^4` nên đệ quy thuần có thể gây `RecursionError`. Ta kiểm tra bằng `sys.getrecursionlimit()`, và sẽ có cách lặp (không đệ quy) ở Cách 4.

In [6]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def chain_tree(n, val=1):
    root = TreeNode(val)
    cur = root
    for _ in range(n - 1):
        cur.left = TreeNode(val)
        cur = cur.left
    return root

def depth_recursive(node):
    return 0 if node is None else 1 + max(depth_recursive(node.left), depth_recursive(node.right))

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    depth_recursive(chain_tree(5000))
except RecursionError:
    print("RecursionError: đệ quy sâu 5000 vượt giới hạn 1000")
finally:
    sys.setrecursionlimit(old)

giới hạn đệ quy hiện tại: 1000
RecursionError: đệ quy sâu 5000 vượt giới hạn 1000


### 3.6. `Counter` và kiểm tra thuộc

`x in danh_sách` tốn `O(độ dài)`; `x in set` hoặc `x in dict` tốn `O(1)`. `collections.Counter` là từ điển đếm số lần xuất hiện, truy cập khóa chưa có trả `0` (không lỗi).

In [7]:
c = Counter([10, 2, 3, 15, 30])
print(c[15], c[7])            # 1 0
print(15 in [10, 2, 3, 15], 15 in c)

1 0
True True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

**Ví dụ 1** `[5,10,10,null,null,2,3]`: tổng `S = 5 + 10 + 10 + 2 + 3 = 30`. Muốn hai phần bằng nhau, mỗi phần phải là `15`. Tổng các cây con (không tính gốc) là `10` (nút trái), `2`, `3`, và `10 + 2 + 3 = 15` (nút phải). Có `15`: **đúng**.

**Ví dụ 2** `[1,2,10,null,null,2,20]`: tổng `S = 1 + 2 + 10 + 2 + 20 = 35` là số lẻ; hai phần nguyên không thể bằng nhau: **sai**.

Trực giác rút ra:

1. Một phần chắc chắn là **một cây con** (phần bị cắt rời), nên chỉ cần xét tổng của các cây con.
2. Nếu tổng của một phần là `x` thì phần kia là `S - x`; bằng nhau khi `x = S / 2`.
3. Vì thế chỉ cần **biết `S` rồi tìm trong các tổng cây con (trừ gốc) giá trị `S / 2`**.
4. Chú ý trường hợp **`S = 0`** (giá trị âm/dương triệt tiêu): khi đó `S / 2 = 0` và phải có một cây con **không phải gốc** có tổng `0`. Nếu ta lỡ tính cả gốc (tổng `0`) thì sẽ nhầm. Ví dụ `[1, -1]`: `S = 0`, gốc có tổng `0` nhưng không có cạnh nào chia đôi bằng nhau (hai phần là `1` và `-1`).

In [8]:
def hand_check(vals):
    root = build_tree(vals)
    sums = [s for _, s in subtree_sums_postorder(root)]
    S = sums[-1]
    non_root = sums[:-1]
    print(vals, "| S =", S, "| tổng cây con (không gốc) =", non_root, "| S/2 =", S / 2)

hand_check([5, 10, 10, None, None, 2, 3])
hand_check([1, 2, 10, None, None, 2, 20])
hand_check([1, -1])
hand_check([0, 0])

[5, 10, 10, None, None, 2, 3] | S = 30 | tổng cây con (không gốc) = [10, 2, 3, 15] | S/2 = 15.0
[1, 2, 10, None, None, 2, 20] | S = 35 | tổng cây con (không gốc) = [2, 2, 20, 32] | S/2 = 17.5
[1, -1] | S = 0 | tổng cây con (không gốc) = [-1] | S/2 = 0.0
[0, 0] | S = 0 | tổng cây con (không gốc) = [0] | S/2 = 0.0


## 5. Cách 1 — Brute Force: thử cắt từng cạnh

### Ý tưởng

Làm đúng theo đề: với **mỗi cạnh**, cắt thật sự, tính tổng hai cây, rồi nối lại.

1. Liệt kê mọi cạnh dưới dạng `(cha, hướng)` với `hướng` là `"left"` hoặc `"right"`.
2. Với mỗi cạnh: gán `cha.hướng = None` (cắt), tính `tổng(root)` và `tổng(con)`; so sánh; gán lại `cha.hướng = con`.
3. Nếu có cạnh nào cho hai tổng bằng nhau thì `True`.

Hàm tính tổng viết bằng ngăn xếp (lặp) để không vướng giới hạn đệ quy.

### Code

In [9]:
def tree_sum(node):
    total = 0
    stack = [node] if node else []
    while stack:
        cur = stack.pop()
        total += cur.val
        if cur.left:
            stack.append(cur.left)
        if cur.right:
            stack.append(cur.right)
    return total


def list_edges(root):
    edges = []
    stack = [root]
    while stack:
        node = stack.pop()
        for side in ("left", "right"):
            child = getattr(node, side)
            if child is not None:
                edges.append((node, side))
                stack.append(child)
    return edges


def check_cut_all(root):
    for parent, side in list_edges(root):
        child = getattr(parent, side)
        setattr(parent, side, None)                  # cắt cạnh
        a, b = tree_sum(root), tree_sum(child)
        setattr(parent, side, child)                 # nối lại
        if a == b:
            return True
    return False

for vals, expected in examples:
    t = build_tree(vals)
    before = tree_to_list(t)
    print(vals, "->", check_cut_all(t), "(kỳ vọng", expected, ")", "| cây nguyên vẹn:", tree_to_list(t) == before)

[5, 10, 10, None, None, 2, 3] -> True (kỳ vọng True ) | cây nguyên vẹn: True
[1, 2, 10, None, None, 2, 20] -> False (kỳ vọng False ) | cây nguyên vẹn: True


### Độ phức tạp và nút thắt

- Time: `n - 1` cạnh, mỗi cạnh tính tổng `O(n)` → `O(n^2)`. Với `n = 10^4` là khoảng `10^8` thao tác nút: quá chậm trong Python.
- Space: `O(n)` cho danh sách cạnh và ngăn xếp.

**Nút thắt:** mỗi lần cắt ta **tính lại** các tổng đã từng tính. Tổng cây con của mọi nút có thể lấy **cùng lúc** trong một lượt duyệt. Hãy xem thời gian tăng thế nào.

In [10]:
for n in [250, 500, 1000, 2000]:
    t = chain_tree(n + 1, 1)                           # tổng n + 1 lẻ: đáp án False nên phải thử hết mọi cạnh
    t0 = time.perf_counter()
    r = check_cut_all(t)
    print(f"n={n}: kết quả {r}, thời gian {time.perf_counter() - t0:.4f}s")

n=250: kết quả False, thời gian 0.0039s
n=500: kết quả False, thời gian 0.0190s
n=1000: kết quả False, thời gian 0.0648s


n=2000: kết quả False, thời gian 0.2845s


Thời gian tăng gấp khoảng 4 lần khi `n` gấp đôi (đúng `O(n^2)`). Ta cố ý chọn dây chuyền có tổng lẻ (đáp án `False`) để phép thử phải đi hết mọi cạnh; nếu đáp án là `True` thì có thể dừng sớm hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá: bài toán chỉ phụ thuộc vào **các tổng cây con** và tổng toàn cây `S`.

> Tồn tại cạnh chia đôi **⇔** tồn tại nút `v` **không phải gốc** với `2 · sub(v) = S`.

Do đó:

1. Một lượt DFS hậu thứ tự tính `sub(v)` cho mọi `v` và **ghi lại** (trừ tổng của gốc).
2. Lấy `S` = tổng của gốc (nút cuối cùng ghi nhận).
3. Nếu `S` lẻ → `False`. Ngược lại trả `S / 2` có nằm trong các tổng đã ghi không.

Vì **gốc được ghi cuối cùng** nên chỉ cần bỏ phần tử cuối (`seen.pop()`) trước khi tra cứu: tổng gốc không bao giờ được xét.

Thời gian `O(n)`: mỗi nút một lần; không gian `O(n)`.

In [11]:
root = build_tree([5, 10, 10, None, None, 2, 3])
seen = []
def dfs(node):
    if node is None:
        return 0
    s = node.val + dfs(node.left) + dfs(node.right)
    seen.append(s)
    return s

S = dfs(root)
print("S =", S, "| seen (hậu thứ tự) =", seen)
seen.pop()                      # bỏ tổng của gốc: không có cạnh nào ứng với gốc
print("sau khi bỏ gốc:", seen, "| S/2 =", S // 2, "| có không?", S // 2 in seen)

S = 30 | seen (hậu thứ tự) = [10, 2, 3, 15, 30]
sau khi bỏ gốc: [10, 2, 3, 15] | S/2 = 15 | có không? True


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Cắt thử từng cạnh, tính lại hai tổng (mục 5) |
| 2 | Một lượt hậu thứ tự lưu mọi tổng cây con trong **danh sách**, bỏ gốc, tra `S/2` (cách nộp) |
| 3 | Lưu các tổng vào **`Counter`** (kể cả gốc), xử lý riêng `S = 0` bằng điều kiện đếm `> 1` |
| 4 | **Bản lặp**: thứ tự duyệt tiền tố + cộng dồn ngược lên cha, không đệ quy |

### Cách 2 — Một lượt hậu thứ tự + danh sách

**Giải thích từng đoạn:**

- `dfs(node)` trả về tổng cây con gốc `node` và ghi nó vào `seen` **sau khi** hai con xong (hậu thứ tự), nên tổng của gốc nằm **cuối** `seen`.
- `s = dfs(root)`: tổng toàn cây. Lẻ → `False` ngay.
- `seen.pop()`: bỏ tổng của gốc (không có cạnh tương ứng).
- `s // 2 in seen`: có cây con (không phải gốc) với tổng `S / 2` hay không.

### Code demo

In [12]:
def check_list(root):
    seen = []

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        seen.append(s)
        return s

    s = dfs(root)
    if s % 2 != 0:
        return False
    seen.pop()
    return s // 2 in seen

for vals, expected in examples:
    print(vals, "->", check_list(build_tree(vals)), "(kỳ vọng", expected, ")")

[5, 10, 10, None, None, 2, 3] -> True (kỳ vọng True )
[1, 2, 10, None, None, 2, 20] -> False (kỳ vọng False )


### Code nộp LeetCode (Cách 2)

```python
class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        seen = []

        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            seen.append(s)
            return s

        s = dfs(root)
        if s % 2 != 0:
            return False
        seen.pop()
        return s // 2 in seen
```

### Độ phức tạp

- Time: `O(n)` để duyệt, cộng thêm `O(n)` cho phép `in` trên danh sách (một lần): tổng `O(n)`.
- Space: `O(n)` cho `seen` và `O(h)` cho đệ quy (`h` là chiều cao, tệ nhất `n`).

### Cách 3 — `Counter` các tổng cây con, xử lý riêng `S = 0`

**Ý tưởng khác biệt:** không bỏ gốc mà đếm **mọi** tổng (kể cả gốc) vào `Counter`. Khi `S != 0` thì `S / 2 != S`, nên tổng của gốc (bằng `S`) không bao giờ bị nhầm với `S / 2`: chỉ cần `cnt[S/2] >= 1`. Khi `S = 0`, gốc tự đóng góp một lần vào `cnt[0]`, nên ta cần **ít nhất hai** phần tử bằng `0` (gốc và một cây con khác): `cnt[0] >= 2`.

Gộp lại: `cnt[S // 2] > (1 if S == 0 else 0)`.

### Code demo

In [13]:
def check_counter(root):
    cnt = Counter()

    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        cnt[s] += 1
        return s

    total = dfs(root)
    if total % 2 != 0:
        return False
    return cnt[total // 2] > (1 if total == 0 else 0)

for vals, expected in examples:
    print(vals, "->", check_counter(build_tree(vals)), "(kỳ vọng", expected, ")")
print("[1, -1] ->", check_counter(build_tree([1, -1])), "(đúng: False)")
print("[0, 0]  ->", check_counter(build_tree([0, 0])), "(đúng: True)")

[5, 10, 10, None, None, 2, 3] -> True (kỳ vọng True )
[1, 2, 10, None, None, 2, 20] -> False (kỳ vọng False )
[1, -1] -> False (đúng: False)
[0, 0]  -> True (đúng: True)


### Code nộp LeetCode (Cách 3)

```python
from collections import Counter

class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        cnt = Counter()

        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            cnt[s] += 1
            return s

        total = dfs(root)
        if total % 2 != 0:
            return False
        return cnt[total // 2] > (1 if total == 0 else 0)
```

### Độ phức tạp

- Time: `O(n)`; tra cứu `Counter` `O(1)`.
- Space: `O(n)` cho bảng đếm và `O(h)` đệ quy.

### Cách 4 — Bản lặp: thứ tự tiền tố + cộng dồn ngược

**Ý tưởng:** tránh đệ quy hoàn toàn.

1. Duyệt bằng ngăn xếp, ghi `order` (mỗi nút xuất hiện **trước** các hậu duệ của nó) và `parent[con] = cha`.
2. Gán `sub[v] = v.val` cho mọi `v`.
3. Duyệt `order` **ngược**: khi tới `v`, mọi hậu duệ của `v` đã được xử lý nên `sub[v]` đã đủ; cộng `sub[v]` vào `sub[parent[v]]`.
4. Cuối cùng `S = sub[root]`; trả `True` nếu có nút `v` không phải gốc với `2 * sub[v] == S`.

So sánh `2 * sub[v] == S` tránh luôn việc chia và kiểm tra lẻ (nếu `S` lẻ thì không có `v` nào thỏa).

### Code demo

In [14]:
def check_iter(root):
    order = []
    parent = {}
    stack = [root]
    while stack:
        node = stack.pop()
        order.append(node)                    # cha luôn đứng trước con trong order
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)

    sub = {node: node.val for node in order}
    for node in reversed(order):              # con được xử lý trước cha
        if node in parent:
            sub[parent[node]] += sub[node]

    total = sub[root]
    return any(2 * sub[node] == total for node in order if node is not root)

for vals, expected in examples:
    print(vals, "->", check_iter(build_tree(vals)), "(kỳ vọng", expected, ")")
print("dây chuyền 10000 nút (không đệ quy):", check_iter(chain_tree(10000, 1)))

[5, 10, 10, None, None, 2, 3] -> True (kỳ vọng True )
[1, 2, 10, None, None, 2, 20] -> False (kỳ vọng False )
dây chuyền 10000 nút (không đệ quy): True


### Code nộp LeetCode (Cách 4)

```python
class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        order, parent = [], {}
        stack = [root]
        while stack:
            node = stack.pop()
            order.append(node)
            for child in (node.left, node.right):
                if child is not None:
                    parent[child] = node
                    stack.append(child)
        sub = {node: node.val for node in order}
        for node in reversed(order):
            if node in parent:
                sub[parent[node]] += sub[node]
        total = sub[root]
        return any(2 * sub[node] == total for node in order if node is not root)
```

### Độ phức tạp

- Time: `O(n)`.
- Space: `O(n)` cho `order`, `parent`, `sub`; không có đệ quy nên không lo `RecursionError` trên cây lệch `10^4` nút.

## 8. TRACE từng bước

Hàm `trace_check(root)` in từng nút lúc **rời** khỏi (sau khi hai con xong): tổng của hai con, tổng cây con của nút, và danh sách `seen` đang có. Cuối cùng in quyết định.

Bảng trạng thái cho **ví dụ 1** `[5,10,10,null,null,2,3]` (thứ tự hậu thứ tự):

| Bước | Nút | `sub(trái)` | `sub(phải)` | `sub(nút)` | `seen` sau bước |
|---|---|---|---|---|---|
| 1 | `10` (trái, lá) | 0 | 0 | 10 | `[10]` |
| 2 | `2` (lá) | 0 | 0 | 2 | `[10, 2]` |
| 3 | `3` (lá) | 0 | 0 | 3 | `[10, 2, 3]` |
| 4 | `10` (phải) | 2 | 3 | 15 | `[10, 2, 3, 15]` |
| 5 | `5` (gốc) | 10 | 15 | 30 | `[10, 2, 3, 15, 30]` |

Kết thúc: `S = 30` chẵn, bỏ gốc (`30`), tìm `15` trong `[10, 2, 3, 15]`: có, nên `True`.

In [15]:
def trace_check(root):
    seen = []

    def dfs(node, depth):
        pad = "  " * depth
        if node is None:
            return 0
        left = dfs(node.left, depth + 1)
        right = dfs(node.right, depth + 1)
        s = node.val + left + right
        seen.append(s)
        print(f"{pad}nút {node.val}: sub(trái)={left}, sub(phải)={right} -> sub={s} | seen = {seen}")
        return s

    S = dfs(root, 0)
    print("S =", S)
    if S % 2 != 0:
        print("S lẻ -> hai phần không thể bằng nhau -> False")
        return False
    popped = seen.pop()
    print(f"bỏ tổng của gốc ({popped}); còn seen = {seen}; cần tìm S/2 = {S // 2}")
    ans = S // 2 in seen
    print("kết quả:", ans)
    return ans

trace_check(build_tree([5, 10, 10, None, None, 2, 3]))

  nút 10: sub(trái)=0, sub(phải)=0 -> sub=10 | seen = [10]
    nút 2: sub(trái)=0, sub(phải)=0 -> sub=2 | seen = [10, 2]
    nút 3: sub(trái)=0, sub(phải)=0 -> sub=3 | seen = [10, 2, 3]
  nút 10: sub(trái)=2, sub(phải)=3 -> sub=15 | seen = [10, 2, 3, 15]
nút 5: sub(trái)=10, sub(phải)=15 -> sub=30 | seen = [10, 2, 3, 15, 30]
S = 30
bỏ tổng của gốc (30); còn seen = [10, 2, 3, 15]; cần tìm S/2 = 15
kết quả: True


True

In [16]:
trace_check(build_tree([1, 2, 10, None, None, 2, 20]))
print()
trace_check(build_tree([1, -1]))       # S = 0: gốc có tổng 0 nhưng bị bỏ đi -> không nhầm

  nút 2: sub(trái)=0, sub(phải)=0 -> sub=2 | seen = [2]
    nút 2: sub(trái)=0, sub(phải)=0 -> sub=2 | seen = [2, 2]
    nút 20: sub(trái)=0, sub(phải)=0 -> sub=20 | seen = [2, 2, 20]
  nút 10: sub(trái)=2, sub(phải)=20 -> sub=32 | seen = [2, 2, 20, 32]
nút 1: sub(trái)=2, sub(phải)=32 -> sub=35 | seen = [2, 2, 20, 32, 35]
S = 35
S lẻ -> hai phần không thể bằng nhau -> False

  nút -1: sub(trái)=0, sub(phải)=0 -> sub=-1 | seen = [-1]
nút 1: sub(trái)=-1, sub(phải)=0 -> sub=0 | seen = [-1, 0]
S = 0
bỏ tổng của gốc (0); còn seen = [-1]; cần tìm S/2 = 0
kết quả: False


False

Quan sát trace thứ ba (`[1, -1]`): `S = 0` (chẵn), `seen` ban đầu là `[-1, 0]`; ta bỏ tổng của **gốc** (`0`) nên còn `[-1]`, mà `S / 2 = 0` không nằm trong đó, trả `False`. Nếu ta quên `seen.pop()` thì `0` xuất hiện và trả sai `True`. Đây chính là lý do phải bỏ gốc.

## 9. Code nộp LeetCode

Ta nộp Cách 2 (một lượt hậu thứ tự + danh sách): ngắn gọn, `O(n)`.

In [17]:
class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        seen = []

        def dfs(node):
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            seen.append(s)
            return s

        s = dfs(root)
        if s % 2 != 0:
            return False
        seen.pop()
        return s // 2 in seen


for vals, expected in examples:
    got = Solution().checkEqualTree(build_tree(vals))
    print(vals, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

[5, 10, 10, None, None, 2, 3] -> True | kỳ vọng: True | OK
[1, 2, 10, None, None, 2, 20] -> False | kỳ vọng: False | OK


## 10. Vì sao đúng

Gọi `S` là tổng toàn cây và, với mỗi nút `v` không phải gốc, `sub(v)` là tổng cây con gốc `v`.

**(a) Mỗi cạnh ứng với đúng một nút không phải gốc.** Cạnh nối `v` với cha của `v`. Cây có `n - 1` cạnh và `n - 1` nút không phải gốc, tương ứng một-một.

**(b) Hậu quả của việc cắt.** Cắt cạnh của `v` tách thành hai cây: cây gốc `v` (tổng `sub(v)`) và phần còn lại chứa gốc (tổng `S - sub(v)`, vì tổng mọi nút là `S`).

**(c) Điều kiện bằng nhau.** `sub(v) = S - sub(v)` ⇔ `2 · sub(v) = S`. Nếu `S` lẻ thì vế trái chẵn, vô nghiệm: `False` ngay. Nếu `S` chẵn thì ⇔ `sub(v) = S / 2`.

**(d) Vì sao phải loại gốc.** Gốc không có cạnh lên cha, nên không ứng với cách cắt nào. `sub(root) = S`, và `S = S / 2` chỉ khi `S = 0`; khi đó nếu không loại gốc ta sẽ báo `True` sai (ví dụ cây `[0]` hoặc `[1, -1]`).

**(e) Giá trị âm không ảnh hưởng.** Lập luận chỉ dùng phép cộng và đẳng thức nên đúng với mọi số nguyên, kể cả âm; ta không dùng tính đơn điệu của tổng.

**(f) Tính đúng của hậu thứ tự.** Khi `dfs(node)` kết thúc thì hai lời gọi con đã trả đúng tổng cây con của chúng (quy nạp theo chiều cao), nên `s` là đúng `sub(node)`; và `seen` chứa mọi `sub(v)`, nút **cuối cùng** được thêm là gốc.

Ta kiểm chứng bằng thí nghiệm: so sánh Cách 2 với phép **cắt thật sự từng cạnh** (Cách 1) trên rất nhiều cây ngẫu nhiên có giá trị nhỏ (để nhiều trường hợp `True` và nhiều tổng bằng `0`).

In [18]:
def random_tree(n, rng, lo=-3, hi=3, deep_bias=0.4):
    # cây ngẫu nhiên n nút, giá trị trong [lo, hi]; deep_bias lớn thì cây lệch/sâu hơn
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        if rng.random() < deep_bias:
            k = len(slots) - 1 - rng.randrange(min(2, len(slots)))
        else:
            k = rng.randrange(len(slots))
        parent, side = slots.pop(k)
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return root


rng = random.Random(663)
trues = 0
for _ in range(3000):
    t = random_tree(rng.randint(1, 20), rng, lo=-3, hi=3, deep_bias=rng.choice([0.1, 0.5, 0.9]))
    expected = check_cut_all(t)
    assert Solution().checkEqualTree(t) == expected, tree_to_list(t)
    trues += expected
print("khớp với cắt thật sự trên 3000 cây; số cây True:", trues, "/ 3000")

khớp với cắt thật sự trên 3000 cây; số cây True: 930 / 3000


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Cắt thử từng cạnh | `O(n^2)` | `O(n)` |
| 2 Một lượt + danh sách | `O(n)` | `O(n)` (+ đệ quy `O(h)`) |
| 3 `Counter` | `O(n)` | `O(n)` (+ đệ quy `O(h)`) |
| 4 Bản lặp | `O(n)` | `O(n)` |

Với `n = 10^4`: Cách 1 cỡ `10^8` bước, các cách còn lại cỡ vài chục nghìn bước. Ta đo thực tế trên cây ngẫu nhiên lớn (Cách 2, 3 cần nâng giới hạn đệ quy cho cây sâu).

In [19]:
sys.setrecursionlimit(30000)

def random_big(n, seed, deep_bias=0.3):
    return random_tree(n, random.Random(seed), lo=-10 ** 5, hi=10 ** 5, deep_bias=deep_bias)

big = {"ngẫu nhiên 10^4": random_big(10000, 1), "lệch sâu 10^4": random_big(10000, 2, 0.98), "dây chuyền 10^4": chain_tree(10000, 1)}
for name, t in big.items():
    out = []
    for fn in (Solution().checkEqualTree, check_counter, check_iter):
        t0 = time.perf_counter()
        r = fn(t)
        out.append((r, round(time.perf_counter() - t0, 4)))
    assert out[0][0] == out[1][0] == out[2][0]
    print(f"{name}: kết quả {out[0][0]} | cách 2 {out[0][1]}s, cách 3 {out[1][1]}s, cách 4 {out[2][1]}s")

t = chain_tree(2000, 1)
t0 = time.perf_counter(); check_cut_all(t); print("Cách 1 trên dây chuyền 2000:", round(time.perf_counter() - t0, 3), "s")

ngẫu nhiên 10^4: kết quả False | cách 2 0.002s, cách 3 0.0037s, cách 4 0.0056s


lệch sâu 10^4: kết quả False | cách 2 0.0016s, cách 3 0.0044s, cách 4 0.0043s


dây chuyền 10^4: kết quả True | cách 2 0.0023s, cách 3 0.004s, cách 4 0.0038s


Cách 1 trên dây chuyền 2000: 0.143 s


## 12. Bộ test

Các nhóm test: ví dụ của đề, edge case (một nút, giá trị `0`, âm, tổng lẻ, tổng `0`), phản ví dụ (các cây làm hỏng code sai), và đối chiếu **nhiều cách** (kể cả cắt thật sự từng cạnh) trên cây ngẫu nhiên có seed, đúng ràng buộc `1 <= n <= 10^4`, `|val| <= 10^5`.

In [20]:
# 1) Ví dụ của đề
for vals, expected in examples:
    assert Solution().checkEqualTree(build_tree(vals)) == expected

# 2) Edge case
cases = {
    (1,): False,                      # một nút: không có cạnh nào để xóa
    (0,): False,                      # một nút giá trị 0: vẫn không có cạnh
    (5,): False,
    (0, 0): True,                     # S = 0, cây con (không gốc) có tổng 0
    (0, 0, 0): True,
    (1, -1): False,                   # S = 0 nhưng hai phần là 1 và -1
    (0, 1, -1): False,                # S = 0, các cây con là 1 và -1
    (2, -1, -1): False,               # S = 0, các cây con là -1 và -1
    (1, 2, 3): True,                  # S = 6, cây con 3
    (1, 2, 4): False,                 # S = 7 lẻ
    (2, 1): False,                    # S = 3 lẻ (3 // 2 = 1 có trong seen nên code ẩu sẽ sai)
    (-2, -1, -1): False,              # S = -4, cần cây con -2 nhưng chỉ có -1, -1
    (0, -3, -3): True,                # S = -6, cây con -3 (số âm vẫn đúng)
    (-1, -1, -1, None, None, -1, -1): False,     # S = -5 lẻ
}
for vals, expected in cases.items():
    t = build_tree(list(vals))
    assert Solution().checkEqualTree(t) == expected, vals
    assert check_cut_all(t) == expected, vals          # oracle cắt thật sự xác nhận bảng đáp án
print("ví dụ + edge OK:", len(cases), "ca (đã đối chiếu oracle cắt thật sự)")

ví dụ + edge OK: 14 ca (đã đối chiếu oracle cắt thật sự)


In [21]:
# 3) Đối chiếu: cắt thật sự (Cách 1) với Cách 2, 3, 4 trên cây nhỏ, giá trị nhỏ (nhiều tổng 0 và nhiều True)
rng = random.Random(663)
trues = zeros = 0
for _ in range(4000):
    t = random_tree(rng.randint(1, 25), rng, lo=rng.choice([-1, -3, -5]), hi=rng.choice([1, 3, 5]), deep_bias=rng.choice([0.0, 0.4, 0.9]))
    a = check_cut_all(t)
    assert Solution().checkEqualTree(t) == check_counter(t) == check_iter(t) == a, tree_to_list(t)
    trues += a
    zeros += (tree_sum(t) == 0)
print("4000 cây: True =", trues, "| số cây có S = 0:", zeros)

# 4) Cây lớn, giá trị lớn (không dùng Cách 1): cả ba cách O(n) phải giống nhau
rng = random.Random(6630)
sys.setrecursionlimit(30000)
for _ in range(40):
    t = random_tree(rng.randint(2000, 10000), rng, lo=-10 ** 5, hi=10 ** 5, deep_bias=rng.choice([0.2, 0.7, 0.99]))
    assert Solution().checkEqualTree(t) == check_counter(t) == check_iter(t)
print("cây lớn: ba cách O(n) khớp")

4000 cây: True = 974 | số cây có S = 0: 142


cây lớn: ba cách O(n) khớp


In [22]:
# 5) Cây lớn mà đáp án chắc chắn True / False (dựng có chủ đích)
def make_true_big(n):
    # dây chuyền gồm hai nửa có tổng bằng nhau: n chẵn, mọi giá trị 1 -> cắt giữa dây chuyền
    return chain_tree(n, 1)

assert check_iter(make_true_big(10000)) is True        # S = 10000, cắt giữa: 5000 + 5000
assert check_iter(make_true_big(9999)) is False         # S lẻ
sys.setrecursionlimit(30000)
assert Solution().checkEqualTree(make_true_big(10000)) is True
assert Solution().checkEqualTree(make_true_big(9999)) is False

# 6) Cây đầu vào không bị sửa
t = build_tree([5, 10, 10, None, None, 2, 3])
before = tree_to_list(t)
Solution().checkEqualTree(t); check_counter(t); check_iter(t)
assert tree_to_list(t) == before
print("cây lớn + cây đầu vào không bị sửa: OK")

cây lớn + cây đầu vào không bị sửa: OK


## 13. Các lỗi người mới hay gặp

Ta cài vài hàm sai cố ý rồi so với oracle (cắt thật sự, Cách 1) để thấy lỗi lộ ra thế nào. `find_mismatch(fn)` tìm cây **nhỏ nhất** mà `fn` trả sai.

In [23]:
def find_mismatch(fn, trials=4000, seed=21, lo=-3, hi=3, min_n=1):
    rng = random.Random(seed)
    best = None
    for _ in range(trials):
        t = random_tree(rng.randint(min_n, 8), rng, lo=lo, hi=hi, deep_bias=rng.choice([0.2, 0.6]))
        if fn(t) != check_cut_all(t):
            lst = tree_to_list(t)
            if best is None or len(lst) < len(best):
                best = lst
    return best

### Lỗi 1: không loại tổng của gốc

Quên `seen.pop()` (hoặc đếm cả gốc). Khi `S = 0` thì `S / 2 = 0` = tổng của gốc, nên bị tính là "có cạnh chia đôi".

In [24]:
def wrong_include_root(root):
    seen = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        seen.append(s)
        return s
    s = dfs(root)
    if s % 2 != 0:
        return False
    return s // 2 in seen                      # SAI: quên seen.pop() bỏ tổng gốc

print("[0]    sai:", wrong_include_root(build_tree([0])), "| đúng:", Solution().checkEqualTree(build_tree([0])))
print("[1,-1] sai:", wrong_include_root(build_tree([1, -1])), "| đúng:", Solution().checkEqualTree(build_tree([1, -1])))
print("cây nhỏ nhất làm sai:", find_mismatch(wrong_include_root))

[0]    sai: True | đúng: False
[1,-1] sai: True | đúng: False
cây nhỏ nhất làm sai: [0]


**Vì sao sai:** gốc không có cạnh để cắt; khi `S = 0`, tổng của gốc (bằng `0`) lọt vào `seen` và bị nhận nhầm là một phần "bằng một nửa". **Code đúng:** `seen.pop()` trước khi tra cứu (hoặc đếm `> 1` khi dùng `Counter`).

### Lỗi 2: không kiểm tra `S` lẻ

Chỉ dùng `S // 2 in seen`. Khi `S` lẻ, `S // 2` bị làm tròn xuống, và nếu số đó tình cờ là tổng của một cây con thì code báo `True` sai.

In [25]:
def wrong_no_parity(root):
    seen = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        seen.append(s)
        return s
    s = dfs(root)
    seen.pop()
    return s // 2 in seen                      # SAI: thiếu kiểm tra s lẻ

print("[2,1] sai:", wrong_no_parity(build_tree([2, 1])), "| đúng:", Solution().checkEqualTree(build_tree([2, 1])))
print("cây nhỏ nhất làm sai:", find_mismatch(wrong_no_parity))

[2,1] sai: True | đúng: False
cây nhỏ nhất làm sai: [1, 0]


**Vì sao sai:** cây `[2, 1]` có `S = 3`, `3 // 2 = 1`, và cây con `1` có tồn tại, nhưng hai phần thật sự là `1` và `2`, không bằng nhau. **Code đúng:** `if s % 2 != 0: return False` (hoặc so sánh `2 * sub == s`).

### Lỗi 3: chỉ xét hai con của gốc

Nhiều người nghĩ "cắt" nghĩa là cắt gốc ra khỏi hai con của nó, nên chỉ so `sub(left)` và `sub(right)` hay `S/2`. Nhưng cạnh có thể nằm **ở sâu** trong cây.

In [26]:
def wrong_only_root_children(root):
    def sub(node):
        return 0 if node is None else node.val + sub(node.left) + sub(node.right)
    s = sub(root)
    if s % 2 != 0:
        return False
    return sub(root.left) == s // 2 or sub(root.right) == s // 2       # SAI: bỏ qua các cạnh sâu hơn

t = build_tree([1, 1, None, 2])             # cạnh cần cắt nằm sâu: cây con 2 ở dưới con trái
print("[1,1,null,2] sai:", wrong_only_root_children(t), "| đúng:", Solution().checkEqualTree(t))
bad = find_mismatch(wrong_only_root_children, min_n=3)
print("cây (>= 3 nút) nhỏ nhất làm sai:", bad)

[1,1,null,2] sai: False | đúng: True
cây (>= 3 nút) nhỏ nhất làm sai: [0, -3, None, -3]


**Vì sao sai:** cạnh cần cắt có thể nằm ở chân một cây con sâu bên dưới (tách ra một cây con có tổng `S / 2` mà không phải con trực tiếp của gốc). **Code đúng:** xét **mọi** nút không phải gốc (qua danh sách `seen`/`Counter`).

### Lỗi 4: đệ quy quá sâu trên cây lệch

Cây `10^4` nút có thể là một dây chuyền sâu `10^4`. Với giới hạn đệ quy mặc định (khoảng `1000`), hàm đệ quy gây `RecursionError`.

In [27]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    Solution().checkEqualTree(chain_tree(10000, 1))
    print("không lỗi")
except RecursionError:
    print("RecursionError với đệ quy sâu 10000 và giới hạn 1000")
finally:
    sys.setrecursionlimit(old)

print("bản lặp (Cách 4) trên cùng cây:", check_iter(chain_tree(10000, 1)))

RecursionError với đệ quy sâu 10000 và giới hạn 1000
bản lặp (Cách 4) trên cùng cây: True


**Vì sao sai (sập):** mỗi nút là một tầng gọi hàm; `10^4` tầng vượt giới hạn mặc định. **Code đúng:** hoặc dùng bản lặp (Cách 4), hoặc nâng giới hạn bằng `sys.setrecursionlimit(...)` (có chi phí bộ nhớ ngăn xếp), hoặc viết duyệt hậu thứ tự bằng ngăn xếp tường minh.

## 14. Mẹo & nhận dạng pattern

- Thấy "xóa một cạnh / chia cây thành hai phần theo tổng" → nghĩ tới **tổng cây con** và **hậu thứ tự**, rồi biến điều kiện về `sub(v) = S / 2`.
- Mỗi cạnh ứng với đúng một nút không phải gốc: **loại gốc** khỏi tập ứng viên.
- Dùng `2 * sub == S` thay vì chia: tránh lỗi làm tròn và xử lý số lẻ/số âm gọn.
- Bài có giá trị âm hoặc `0` → luôn thử `S = 0` và các tổng âm; đừng dựa vào tính đơn điệu.
- Khi lo `RecursionError` (cây `10^4`, lệch) → bản lặp với `order` tiền tố + cộng dồn ngược, hoặc nâng giới hạn.
- Cùng mẫu "tính một đại lượng cây con rồi so với tổng": LeetCode 1339, 508, 2265, 1120.
- Luôn tự viết oracle "cắt thật sự" cho cây nhỏ để đối chiếu.

## 15. Biến thể

### 15.1. Đếm số cách cắt hợp lệ

Nếu hỏi "có bao nhiêu cạnh cho hai phần bằng nhau?" thì đếm số nút không phải gốc có `2 * sub == S`.

In [28]:
def count_valid_cuts(root):
    order, parent = [], {}
    stack = [root]
    while stack:
        node = stack.pop()
        order.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    sub = {node: node.val for node in order}
    for node in reversed(order):
        if node in parent:
            sub[parent[node]] += sub[node]
    total = sub[root]
    return sum(1 for node in order if node is not root and 2 * sub[node] == total)

def count_valid_cuts_oracle(root):
    cnt = 0
    for parent, side in list_edges(root):
        child = getattr(parent, side)
        setattr(parent, side, None)
        cnt += tree_sum(root) == tree_sum(child)
        setattr(parent, side, child)
    return cnt

rng = random.Random(151)
for _ in range(500):
    t = random_tree(rng.randint(1, 20), rng, lo=-2, hi=2, deep_bias=0.5)
    assert count_valid_cuts(t) == count_valid_cuts_oracle(t)
print("số cạnh hợp lệ trong [0,0,0,0]:", count_valid_cuts(build_tree([0, 0, 0, 0])))

số cạnh hợp lệ trong [0,0,0,0]: 3


### 15.2. Tích lớn nhất của hai phần khi cắt một cạnh (LeetCode 1339)

Cắt một cạnh sao cho tích hai tổng `sub(v) * (S - sub(v))` **lớn nhất** (lấy modulo `10^9 + 7` ở bước cuối). Cùng mẫu: duyệt tổng cây con, nhưng thay "tìm `S/2`" bằng "tối ưu hóa". Chú ý so sánh bằng số nguyên **chưa** lấy modulo.

In [29]:
def max_product_split(root, mod=10 ** 9 + 7):
    sums = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        sums.append(s)
        return s
    S = dfs(root)
    sums.pop()                              # bỏ gốc
    return max(x * (S - x) for x in sums) % mod

t = build_tree([1, 2, 3, 4, 5, 6])
print("max_product_split([1,2,3,4,5,6]) =", max_product_split(t), "(đề LeetCode 1339 cho 110)")

max_product_split([1,2,3,4,5,6]) = 110 (đề LeetCode 1339 cho 110)


### 15.3. Bài liên quan

- LeetCode 508 *Most Frequent Subtree Sum*: đếm tổng cây con bằng `Counter` (đúng Cách 3).
- LeetCode 1339 *Maximum Product of Splitted Binary Tree*.
- LeetCode 563 *Binary Tree Tilt*, 543 *Diameter of Binary Tree*: "trả về một đại lượng cây con, cập nhật đáp án toàn cục".
- LeetCode 416 *Partition Equal Subset Sum*: cũng là "chia đôi tổng", nhưng trên tập con (quy hoạch động).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Cắt thử từng cạnh | cắt thật rồi tính lại hai tổng | `O(n^2)` | `O(n)` | oracle kiểm thử cây nhỏ |
| 2 Một lượt + danh sách | lưu tổng cây con, bỏ gốc, tra `S/2` | `O(n)` | `O(n)` | **cách nộp**, ngắn nhất |
| 3 `Counter` | đếm tổng, `S = 0` cần đếm `> 1` | `O(n)` | `O(n)` | khi cần đếm số lần/ nhiều truy vấn |
| 4 Bản lặp | thứ tự tiền tố + cộng dồn ngược | `O(n)` | `O(n)` | cây lệch rất sâu, tránh `RecursionError` |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Hàm `subtree_sums_postorder` (mục 3.2) trả danh sách `(val, tổng cây con)` theo hậu thứ tự. Đoán **chỉ phần tổng** theo hậu thứ tự cho hai cây rồi điền `ex1_answer` (danh sách hai danh sách). **Không** chạy trước khi đoán.

1. `[1, 2, 3, 4, 5]` (gốc `1`, con trái `2` có hai con `4`, `5`; con phải `3`)
2. `[3, -1, -2]`

In [30]:
ex1_trees = [[1, 2, 3, 4, 5], [3, -1, -2]]
ex1_answer = None       # ví dụ [[1, 2, 3], [4, 5]]  <- chỉ là mẫu định dạng

### Bài 2 (tính tay)

Không chạy code, cho biết `checkEqualTree` trả `True` hay `False` với từng cây: `[1, 2, 3]`, `[1, 2, 4]`, `[0, 1, -1]`, `[0, 0]`. Điền `ex2_answer` là danh sách 4 giá trị `bool`.

In [31]:
ex2_trees = [[1, 2, 3], [1, 2, 4], [0, 1, -1], [0, 0]]
ex2_answer = None       # ví dụ [True, True, True, True]  <- chỉ là mẫu định dạng

### Bài 3 (điền TODO)

Hoàn thành `ex3_check`: hai chỗ `TODO` là điều kiện trả `False` sớm khi tổng lẻ, và phép tra cứu cuối cùng. Cell này **không gọi** hàm.

In [32]:
def ex3_check(root):
    seen = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        seen.append(s)
        return s
    s = dfs(root)
    # TODO 1: nếu tổng lẻ thì trả False
    seen.pop()
    # TODO 2: trả về True nếu S/2 có trong seen
    return None

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(root)` bằng bản **lặp** (Cách 4). Để `pass` nếu chưa làm.

In [33]:
def ex4_rewrite(root):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_min_diff(root)`: xóa đúng một cạnh sao cho **chênh lệch tuyệt đối** giữa tổng hai phần nhỏ nhất; trả về chênh lệch đó (hoặc `None` nếu cây chỉ có một nút). Ví dụ `[5,10,10,null,null,2,3]` cho `0`.

In [34]:
def ex5_min_diff(root):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: hậu thứ tự: trái, phải, rồi nút; ghi tổng cây con của nút ngay khi xong hai con.
- Mức 2: lá ghi chính giá trị của nó; nút trong ghi `val + tổng trái + tổng phải`.
- Mức 3: cây 1: `4, 5, 2 + 4 + 5 = 11, 3, 1 + 11 + 3 = 15`. Cây 2: `-1, -2, 3 + (-1) + (-2) = 0`.

**Bài 2**
- Mức 1: tính `S`; lẻ thì `False`.
- Mức 2: tìm cây con (không phải gốc) có tổng `S / 2`.
- Mức 3: `[1,2,3]`: `S = 6`, cây con `3` có: `True`. `[1,2,4]`: `S = 7` lẻ: `False`. `[0,1,-1]`: `S = 0`, các cây con `1`, `-1`: không có `0`: `False`. `[0,0]`: `S = 0`, cây con `0` (không phải gốc): `True`.

**Bài 3**
- Mức 1: `S` lẻ thì không thể chia đôi.
- Mức 2: TODO 1 dùng `s % 2 != 0`; TODO 2 là `s // 2 in seen` (sau khi đã `pop` gốc).
- Mức 3: `if s % 2 != 0: return False` và `return s // 2 in seen`.

**Bài 4**
- Mức 1: tạo `order` và `parent` bằng ngăn xếp.
- Mức 2: duyệt `order` ngược và cộng `sub[node]` vào `sub[parent[node]]`.
- Mức 3: kết quả `any(2 * sub[v] == sub[root] for v in order if v is not root)`.

**Bài 5**
- Mức 1: cắt cạnh của `v` cho hai phần `sub(v)` và `S - sub(v)`.
- Mức 2: chênh lệch là `abs(S - 2 * sub(v))`; lấy min trên mọi `v` không phải gốc.
- Mức 3: dùng bản lặp hoặc đệ quy; trả `None` nếu chỉ có một nút.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [35]:
ex1_solution = [[4, 5, 11, 3, 15], [-1, -2, 0]]
assert [[s for _, s in subtree_sums_postorder(build_tree(v))] for v in ex1_trees] == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: [[4, 5, 11, 3, 15], [-1, -2, 0]]


### Đáp án Bài 2

In [36]:
ex2_solution = [True, False, False, True]
assert [Solution().checkEqualTree(build_tree(v)) for v in ex2_trees] == ex2_solution
assert [check_cut_all(build_tree(v)) for v in ex2_trees] == ex2_solution        # oracle cắt thật sự
print("Bài 2 OK:", ex2_solution)

Bài 2 OK: [True, False, False, True]


### Đáp án Bài 3

In [37]:
def ex3_solution(root):
    seen = []
    def dfs(node):
        if node is None:
            return 0
        s = node.val + dfs(node.left) + dfs(node.right)
        seen.append(s)
        return s
    s = dfs(root)
    if s % 2 != 0:
        return False
    seen.pop()
    return s // 2 in seen

rng = random.Random(3)
for _ in range(500):
    t = random_tree(rng.randint(1, 18), rng, lo=-3, hi=3, deep_bias=0.5)
    assert ex3_solution(t) == check_cut_all(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [38]:
def ex4_solution(root):
    order, parent = [], {}
    stack = [root]
    while stack:
        node = stack.pop()
        order.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    sub = {node: node.val for node in order}
    for node in reversed(order):
        if node in parent:
            sub[parent[node]] += sub[node]
    return any(2 * sub[node] == sub[root] for node in order if node is not root)

rng = random.Random(4)
for _ in range(500):
    t = random_tree(rng.randint(1, 18), rng, lo=-3, hi=3, deep_bias=0.5)
    assert ex4_solution(t) == check_cut_all(t)
assert ex4_solution(chain_tree(10000, 1)) is True
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
def ex5_solution(root):
    order, parent = [], {}
    stack = [root]
    while stack:
        node = stack.pop()
        order.append(node)
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    sub = {node: node.val for node in order}
    for node in reversed(order):
        if node in parent:
            sub[parent[node]] += sub[node]
    total = sub[root]
    diffs = [abs(total - 2 * sub[node]) for node in order if node is not root]
    return min(diffs) if diffs else None

def ex5_oracle(root):
    best = None
    for parent, side in list_edges(root):
        child = getattr(parent, side)
        setattr(parent, side, None)
        d = abs(tree_sum(root) - tree_sum(child))
        setattr(parent, side, child)
        best = d if best is None else min(best, d)
    return best

assert ex5_solution(build_tree([5, 10, 10, None, None, 2, 3])) == 0
assert ex5_solution(TreeNode(7)) is None
rng = random.Random(5)
for _ in range(500):
    t = random_tree(rng.randint(1, 20), rng, lo=-5, hi=5, deep_bias=0.5)
    assert ex5_solution(t) == ex5_oracle(t)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def checkEqualTree(self, root: Optional[TreeNode]) -> bool:
        seen = []

        def dfs(node):                                   # hậu thứ tự: tổng cây con
            if node is None:
                return 0
            s = node.val + dfs(node.left) + dfs(node.right)
            seen.append(s)
            return s

        s = dfs(root)                                    # gốc được ghi cuối cùng
        if s % 2 != 0:                                   # tổng lẻ: không thể chia đôi
            return False
        seen.pop()                                       # bỏ gốc: không có cạnh tương ứng
        return s // 2 in seen
```

Câu thần chú: **"tổng cây con, bỏ gốc, tổng lẻ thì sai, tìm S chia hai."**

### Checklist trước khi submit

- [ ] Duyệt **hậu thứ tự**: ghi `sub(node)` sau khi hai con xong.
- [ ] **Loại gốc** (không có cạnh): `seen.pop()` hoặc điều kiện `node is not root` / đếm `> 1` khi `S = 0`.
- [ ] Kiểm tra `S` lẻ (`s % 2 != 0`) trước khi dùng `S // 2` (hoặc dùng `2 * sub == S`).
- [ ] Đã nghĩ tới `S = 0`, giá trị `0` và số âm (`[0]`, `[0,0]`, `[1,-1]`).
- [ ] Cây một nút trả `False`.
- [ ] Cây lệch `10^4` nút: đệ quy có thể `RecursionError` ở môi trường giới hạn; có sẵn bản lặp.
- [ ] Đối chiếu bằng oracle cắt thật sự trên cây nhỏ ngẫu nhiên.
- [ ] Skeleton đúng: `class Solution`, method `checkEqualTree(self, root: Optional[TreeNode]) -> bool`.